In [6]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics     import roc_auc_score
from torch_geometric.data import DataLoader, InMemoryDataset, Data
import h5py

from pathlib import Path
from torch_geometric.nn import global_max_pool, global_mean_pool, global_add_pool, GINConv, EdgeConv
from torch.nn import Linear, BatchNorm1d, ReLU
import joblib

PATH = "../../.."
# PATH = "storage"

In [7]:
class SlideGraphArch(torch.nn.Module):
    """Define SlideGraph architecture."""

    def __init__(
        self,
        dim_features: int,
        dim_target: int,
        layers: tuple[int, int] | None = None,
        pooling: str = "max",
        dropout: float = 0.0,
        conv: str = "GINConv",
        *,
        gembed: bool = False,
        **kwargs,
    ):
        super().__init__()
        if layers is None:
            layers = [6, 6]
        self.dropout = dropout
        self.embeddings_dim = layers
        self.num_layers = len(layers)
        self.pooling = {
            "max": global_max_pool,
            "mean": global_mean_pool,
            "add": global_add_pool,
        }[pooling]
        self.gembed = gembed

        conv_dict = {"GINConv": [GINConv, 1], "EdgeConv": [EdgeConv, 2]}
        if conv not in conv_dict:
            raise ValueError(f'Unsupported conv="{conv}"')

        def create_lin(i, o):
            return torch.nn.Sequential(Linear(i, o), BatchNorm1d(o), ReLU())

        # first embedding layer
        input_dim = dim_features
        out_dim = layers[0]
        self.first_h = create_lin(input_dim, out_dim)
        self.linears = torch.nn.ModuleList([Linear(out_dim, dim_target)])

        input_dim = out_dim

        # subsequent conv + linear layers
        self.nns, self.convs = [], []
        for od in layers[1:]:
            conv_cls, alpha = conv_dict[conv]
            subnet = create_lin(alpha * input_dim, od)
            self.nns.append(subnet)
            self.convs.append(conv_cls(subnet, **kwargs))
            self.linears.append(Linear(od, dim_target))
            input_dim = od

        self.nns = torch.nn.ModuleList(self.nns)
        self.convs = torch.nn.ModuleList(self.convs)

        # aux holds any external models you want to save
        self.aux_model = {}

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.first_h(x)
        wsi_pred = 0
        # layer 0
        node_score = self.linears[0](x)
        pooled = self.pooling(node_score, batch)
        wsi_pred = F.dropout(pooled, p=self.dropout, training=self.training)
        # deeper layers
        for i in range(1, self.num_layers):
            x = self.convs[i-1](x, edge_index)
            if not self.gembed:
                node_score = self.linears[i](x)
                pooled = self.pooling(node_score, batch)
                part = F.dropout(pooled, p=self.dropout, training=self.training)
            else:
                pooled = self.pooling(x, batch)
                part = F.dropout(self.linears[i](pooled), p=self.dropout, training=self.training)
            wsi_pred = wsi_pred + part
        return wsi_pred, node_score

    @staticmethod
    def train_batch(model, batch, optimizer, device):
        """
        batch: a PyG Batch with .x, .edge_index, .batch and .y
        """
        batch = batch.to(device)
        model.train()
        optimizer.zero_grad()
        out, _ = model(batch)                  # out: [batch_size, dim_target]
        labels = batch.y.view(-1).to(device)   # slide-level labels
        loss = F.binary_cross_entropy_with_logits(out.view(-1), labels)
        loss.backward()
        optimizer.step()
        return loss.item(), out.detach().cpu(), labels.cpu()

    @staticmethod
    def infer_batch(model, batch, device):
        """
        batch: a PyG Batch with .x, .edge_index, .batch and .y
        """
        batch = batch.to(device)
        model.eval()
        with torch.inference_mode():
            out, _ = model(batch)
        labels = batch.y.view(-1).cpu()
        return out.cpu(), labels

    def save(self, path, aux_path):
        torch.save(self.state_dict(), path)
        joblib.dump(self.aux_model, aux_path)

    def load(self, path, aux_path):
        sd = torch.load(path, map_location='cpu')
        self.load_state_dict(sd)
        self.aux_model = joblib.load(aux_path)

# ───────────────────────────────────────────────────────────

import os
import h5py
import torch
import sklearn.neighbors
from pathlib import Path
from torch_geometric.data import InMemoryDataset, Data

class WSIGraphDataset(InMemoryDataset):
    """
    InMemoryDataset that reads CAMELYON/{train,test}/feats/features_{encoder}/*.h5,
    builds k-NN graphs from coords, and pulls slide-level label from the HDF5 attrs.
    Slide IDs are kept as the HDF5 filename stem (e.g. 'test_001').
    """
    def __init__(self, root: str, encoder_name: str, k_nn: int = 8):
        # Store these before calling super(), so process() can access them
        self.root_dir     = Path(root)
        self.encoder_name = encoder_name
        self.k_nn         = k_nn

        # Initialize parent (this may call self.process())
        super().__init__(root)

        # After super, either load processed or run process()
        processed_path = Path(self.processed_paths[0])
        if processed_path.exists():
            # <-- pass weights_only=False so torch_geometric Data can unpickle
            self.data, self.slices = torch.load(processed_path, weights_only=False)
        else:
            self.process()

    @property
    def processed_file_names(self):
        return ["wsi_graphs.pt"]

    def process(self):
        data_list = []
        for split in ["train", "test"]:
            feats_dir = self.root_dir / split / "feats" / f"features_{self.encoder_name}"
            if not feats_dir.exists():
                continue

            for h5name in os.listdir(feats_dir):
                if not h5name.lower().endswith(".h5"):
                    continue

                sample_id = Path(h5name).stem  # e.g. 'test_001'
                h5path    = feats_dir / h5name
                with h5py.File(h5path, "r") as h5:
                    coords = h5["coords"][()]    # (N,2)
                    feats  = h5["features"][()]  # (N,D)
                    label  = float(h5.attrs.get("label", 0))

                # Build k-NN edge index
                nbrs = sklearn.neighbors.NearestNeighbors(n_neighbors=self.k_nn).fit(coords)
                _, idx = nbrs.kneighbors(coords)
                edges = [
                    [i, j]
                    for i, neighbors in enumerate(idx)
                    for j in neighbors
                    if i != j
                ]
                edge_index = torch.tensor(edges, dtype=torch.long).T

                data_list.append(
                    Data(
                        x=torch.tensor(feats, dtype=torch.float32),
                        edge_index=edge_index,
                        y=torch.tensor([label], dtype=torch.float32),
                        slide_id=sample_id
                    )
                )

        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

In [8]:
# 1) Get indices for a given split (train or test)
def get_split_indices(dataset, mapping_csv, split_name):
    df = pd.read_csv(mapping_csv, dtype={'sample_id':str})
    df = df[df['set'] == split_name]
    want = set(df['sample_id'].tolist())
    return [i for i, d in enumerate(dataset) if d.slide_id in want]

def stratified_train_val_split(dataset, train_indices, mapping_csv, val_frac=0.2, random_state=42):
    df = pd.read_csv(mapping_csv, dtype={'sample_id':str}).set_index('sample_id')
    labels = [int(df.loc[dataset[i].slide_id, 'label']) for i in train_indices]
    tr, val = train_test_split(
        train_indices,
        test_size=val_frac,
        stratify=labels,
        random_state=random_state
    )
    return tr, val

# 3) Training loop that takes DataLoaders as inputs
def train_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    max_epochs=20,
    patience=5
):
    """
    model:        SlideGraphArch instance
    train_loader: DataLoader on training graphs
    val_loader:   DataLoader on validation graphs
    """
    best_val = float('inf')
    no_imp   = 0
    best_state = None

    for epoch in range(1, max_epochs+1):
        # — Training —
        model.train()
        tr_losses = []
        for batch in train_loader:
            loss, _, _ = SlideGraphArch.train_batch(model, batch, optimizer, device)
            tr_losses.append(loss)
        avg_tr = np.mean(tr_losses)

        # — Validation —
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                out, lbl = SlideGraphArch.infer_batch(model, batch, device)
                val_losses.append(
                    F.binary_cross_entropy_with_logits(out.view(-1), lbl).item()
                )
        avg_val = np.mean(val_losses)

        print(f"Epoch {epoch}: Train Loss={avg_tr:.4f}, Val Loss={avg_val:.4f}")

        # — Early stopping check —
        if avg_val < best_val:
            best_val = avg_val
            best_state = model.state_dict()
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print("Early stopping.")
                break

    # restore best weights
    model.load_state_dict(best_state)
    return model

In [9]:
# ───────────────────────────────────────────────────────────
# 4) Example usage: build your loaders and call train_loop
# ───────────────────────────────────────────────────────────

# a) Build the full dataset once
root    = os.path.join(PATH, "Data", "CAMELYON")
encoder = "GigaPath"
ds      = WSIGraphDataset(root, encoder_name=encoder, k_nn=8)

# b) Get train/test indices
mapping_csv = os.path.join(PATH, "Data", "CAMELYON", "class_mapping.csv")
train_indices = get_split_indices(ds, mapping_csv, "train")
test_indices  = get_split_indices(ds, mapping_csv, "test")

# c) Split train → train/val
train_idx, val_idx = stratified_train_val_split(ds, train_indices, mapping_csv, val_frac=0.3)

# d) Create DataLoaders
batch_size = 8
train_loader = DataLoader([ds[i] for i in train_idx], batch_size=batch_size, shuffle=True)
val_loader   = DataLoader([ds[i] for i in val_idx],   batch_size=batch_size)
test_loader  = DataLoader([ds[i] for i in test_indices], batch_size=batch_size)

# e) Instantiate your GNN and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feat_dim = ds[0].x.size(1)
model = SlideGraphArch(dim_features=feat_dim, dim_target=1,
                       layers=(512,256), pooling="mean",
                       dropout=0.2, conv="GINConv", gembed=False).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)

Processing...
Done!
c:\Users\Dhyey\anaconda3\envs\trident\lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [10]:
# for i, batch in enumerate(train_loader):
#     print(f"Batch {i}:")
#     labels = batch.y.view(-1).tolist()
    
#     slide_ids = batch.to_data_list()
    
#     for j, sample in enumerate(slide_ids):
#         slide_name = sample.slide_id
#         label = sample.y.item()
#         print(f"  {slide_name}: {label}")
#     print("---")

from collections import Counter
import torch

def count_classes_in_loader(loader):
    class_counter = Counter()

    for batch in loader:
        labels = batch.y.view(-1)  # Flatten in case it's [B, 1]
        for label in labels:
            class_counter[int(label.item())] += 1  # Convert tensor -> int
    
    return dict(class_counter)

train_counts = count_classes_in_loader(train_loader)
val_counts   = count_classes_in_loader(val_loader)
test_counts  = count_classes_in_loader(test_loader)

print("Train class counts:", train_counts)
print("Val class counts:", val_counts)
print("Test class counts:", test_counts)


Train class counts: {1: 78, 0: 83}
Val class counts: {0: 36, 1: 33}
Test class counts: {1: 49, 0: 80}


In [11]:
# f) Train with early stopping
model = train_loop(model, train_loader, val_loader, optimizer, device,
                   max_epochs=1000, patience=500)

Epoch 1: Train Loss=0.7010, Val Loss=0.6829
Epoch 2: Train Loss=0.6323, Val Loss=0.6608
Epoch 3: Train Loss=0.6291, Val Loss=0.6868
Epoch 4: Train Loss=0.6023, Val Loss=0.6817
Epoch 5: Train Loss=0.5933, Val Loss=0.6734
Epoch 6: Train Loss=0.5845, Val Loss=0.6816
Epoch 7: Train Loss=0.5404, Val Loss=0.6707
Epoch 8: Train Loss=0.5917, Val Loss=0.6649
Epoch 9: Train Loss=0.5429, Val Loss=0.7001
Epoch 10: Train Loss=0.5613, Val Loss=0.6680
Epoch 11: Train Loss=0.5451, Val Loss=0.6725
Epoch 12: Train Loss=0.5561, Val Loss=0.6568
Epoch 13: Train Loss=0.5394, Val Loss=0.6737
Epoch 14: Train Loss=0.5358, Val Loss=0.6812
Epoch 15: Train Loss=0.5366, Val Loss=0.6573
Epoch 16: Train Loss=0.5470, Val Loss=0.6758
Epoch 17: Train Loss=0.5141, Val Loss=0.6841
Epoch 18: Train Loss=0.5016, Val Loss=0.6690
Epoch 19: Train Loss=0.4962, Val Loss=0.6555
Epoch 20: Train Loss=0.4767, Val Loss=0.6668
Epoch 21: Train Loss=0.4699, Val Loss=0.6609
Epoch 22: Train Loss=0.4672, Val Loss=0.6693
Epoch 23: Train Los

In [12]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
import numpy as np

model.eval()
all_preds, all_lbls = [], []
with torch.no_grad():
    for batch in test_loader:
        out, lbl = SlideGraphArch.infer_batch(model, batch, device)
        all_preds.append(out.cpu().numpy())
        all_lbls.append(lbl.cpu().numpy())
all_preds = np.concatenate(all_preds).ravel()
all_lbls  = np.concatenate(all_lbls).ravel()

# Binary predictions (threshold at 0.0 for logits, or 0.5 if using sigmoid)
probs = 1 / (1 + np.exp(-all_preds))  # convert logits to probs
y_pred = (probs >= 0.5).astype(int)

# Compute stats
auc  = roc_auc_score(all_lbls, probs)
ap   = average_precision_score(all_lbls, probs)
acc  = accuracy_score(all_lbls, y_pred)
prec = precision_score(all_lbls, y_pred)
rec  = recall_score(all_lbls, y_pred)
f1   = f1_score(all_lbls, y_pred)
cm   = confusion_matrix(all_lbls, y_pred)

print(f"Test ROC AUC        = {auc:.4f}")
print(f"Test Average Precision (AUPRC) = {ap:.4f}")
print(f"Test Accuracy       = {acc:.4f}")
print(f"Test Precision      = {prec:.4f}")
print(f"Test Recall         = {rec:.4f}")
print(f"Test F1 Score       = {f1:.4f}")
print(f"Confusion Matrix:")
print(cm)

Test ROC AUC        = 0.5837
Test Average Precision (AUPRC) = 0.4431
Test Accuracy       = 0.5271
Test Precision      = 0.4250
Test Recall         = 0.6939
Test F1 Score       = 0.5271
Confusion Matrix:
[[34 46]
 [15 34]]


In [8]:
import h5py

def inspect_h5_with_label(h5_path, print_values=False, max_items=5):
    """
    Inspect an HDF5 feature file and print:
      - file-level attributes (including 'label' if present)
      - for each dataset: name, shape, dtype, attributes
      - optionally, a small sample of each dataset’s values
    """
    print(f"Inspecting: {h5_path}\n")
    with h5py.File(h5_path, 'r') as f:
        # 1) File‐level attrs
        if f.attrs:
            print("File-level attributes:")
            for k, v in f.attrs.items():
                print(f"  {k}: {v}")
        else:
            print("No file-level attributes found.")
        print()

        # 2) Datasets and Groups
        def _print_item(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"[Dataset] {name}")
                print(f"  shape: {obj.shape}, dtype: {obj.dtype}")
                if obj.attrs:
                    print("  attributes:")
                    for ak, av in obj.attrs.items():
                        print(f"    {ak}: {av}")
                if print_values:
                    try:
                        data = obj[()]
                        flat = data.ravel()
                        preview = flat[:max_items]
                        print(f"  sample values: {preview}")
                    except Exception as e:
                        print(f"  could not read values: {e}")
                print()
            elif isinstance(obj, h5py.Group):
                print(f"[Group] {name}/")
                print()

        f.visititems(_print_item)

# Example usage:
inspect_h5_with_label(
    r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\test\feats\features_midnight12k\test_030.h5",
    print_values=True,
    max_items=10
)

Inspecting: C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\test\feats\features_midnight12k\test_030.h5

File-level attributes:
  label: 1

[Dataset] coords
  shape: (750, 2), dtype: int64
  attributes:
    level0_magnification: 40
    name: test_030
    overlap: 0
    patch_size: 512
    patch_size_level0: 2048
    savetodir: storage/data/CAMELYON/test/feats
    target_magnification: 10
  sample values: [ 14336 151552  14336 153600  14336 155648  14336 157696  14336 159744]

[Dataset] features
  shape: (750, 1536), dtype: float32
  attributes:
    name: test_030
    savetodir: storage/data/CAMELYON/test/feats/features_midnight12k
  sample values: [ 0.05745624 -0.1913878   0.03104649 -0.00160325 -0.13733384  0.16951458
  0.09284064  0.32994306  0.0426835   1.0957878 ]

